In [ ]:
# =============================
# IMPORTS
# =============================
import pandas as pd
import numpy as np
import yfinance as yf
import requests
from datetime import datetime, timedelta
import logging
import time

logging.getLogger("yfinance").setLevel(logging.CRITICAL)

# =============================
# SETTINGS
# =============================
LOOKBACK_DAYS = 7

MIN_PRICE = 25
MAX_PRICE = 500

MAX_VOLATILITY = 0.07
MIN_TREND = 0.0

# Filters
USE_REACTION_FILTER = True
USE_SPIKE_FILTER = True

MAX_EARNINGS_MOVE = 0.15  # 15%

# =============================
# DATE SETUP
# =============================
today = datetime.now().date()

if today.weekday() == 5:
    today -= timedelta(days=1)
elif today.weekday() == 6:
    today -= timedelta(days=2)

start_date = today - timedelta(days=LOOKBACK_DAYS)

print(f"\n📅 Earnings window: {start_date} → {today}")

# =============================
# NASDAQ EARNINGS FETCH
# =============================
def get_earnings_for_day(date):
    url = f"https://api.nasdaq.com/api/calendar/earnings?date={date}"
    headers = {"User-Agent": "Mozilla/5.0"}
    try:
        r = requests.get(url, headers=headers)
        data = r.json()
        return [row["symbol"] for row in data["data"]["rows"]]
    except:
        return []

earnings_tickers = []
current = start_date

print("\n🔎 Fetching earnings...")

while current <= today:
    daily = get_earnings_for_day(current.strftime("%Y-%m-%d"))
    print(f"{current}: {len(daily)} stocks")
    earnings_tickers.extend(daily)
    current += timedelta(days=1)
    time.sleep(0.3)

earnings_tickers = list(set(earnings_tickers))

print(f"\n✅ Total earnings stocks: {len(earnings_tickers)}")

# =============================
# FILTERING
# =============================
results = []

counts = {
    "start": len(earnings_tickers),
    "price": 0,
    "trend": 0,
    "volatility": 0,
    "final": 0
}

print("\n⚙️ Running filters...\n")

for symbol in earnings_tickers:
    try:
        stock = yf.Ticker(symbol)
        hist = stock.history(period="3mo")

        if len(hist) < 50:
            continue

        price = hist["Close"].iloc[-1]

        # PRICE
        if price < MIN_PRICE or price > MAX_PRICE:
            continue
        counts["price"] += 1

        # EARNINGS FILTERS
        if USE_REACTION_FILTER or USE_SPIKE_FILTER:
            recent = hist.tail(10)
            returns = recent["Close"].pct_change()

            idx = returns.abs().idxmax()

            try:
                pre_price = hist.loc[:idx].iloc[-2]["Close"]
                post_price = hist.loc[idx]["Close"]
            except:
                continue

            move_pct = (post_price - pre_price) / pre_price

            if USE_SPIKE_FILTER:
                if abs(move_pct) > MAX_EARNINGS_MOVE:
                    continue

            if USE_REACTION_FILTER:
                if post_price < pre_price:
                    continue

        # TREND
        hist["SMA20"] = hist["Close"].rolling(20).mean()
        hist["SMA50"] = hist["Close"].rolling(50).mean()

        if pd.isna(hist["SMA50"].iloc[-1]):
            continue

        trend = (hist["SMA20"].iloc[-1] - hist["SMA50"].iloc[-1]) / hist["SMA50"].iloc[-1]

        if trend < MIN_TREND:
            continue
        counts["trend"] += 1

        # VOLATILITY
        vol = hist["Close"].pct_change().std()

        if vol > MAX_VOLATILITY:
            continue
        counts["volatility"] += 1

        # =============================
        # SPREAD SELECTOR (NEW)
        # =============================
        spread = 2.5  # default

        strong_trend = trend > 0.03   # ~3% slope
        clean_reaction = True if not USE_REACTION_FILTER else (post_price >= pre_price)

        if strong_trend and clean_reaction and vol > 0.02:
            spread = 5

        # OUTPUT
        info = stock.info
        sector = info.get("sector", "Unknown")

        results.append({
            "Ticker": symbol,
            "Sector": sector,
            "Price": round(price, 2),
            "Trend %": round(trend * 100, 2),
            "Volatility": round(vol, 4),
            "Suggested Spread": f"${spread}"
        })

        counts["final"] += 1

    except:
        continue

# =============================
# DEBUG
# =============================
print("\n📊 FILTER PROGRESSION:\n")
for k, v in counts.items():
    print(f"{k.capitalize():<12}: {v}")

# =============================
# RESULTS
# =============================
df = pd.DataFrame(results)

if df.empty:
    print("\n❌ No valid setups this cycle")
else:
    df_sorted = df.sort_values(by="Trend %", ascending=False)

    print("\n✅ TOP CANDIDATES:\n")
    print(df_sorted.head(15))

    print("\n📋 COPY LIST:\n")
    print(", ".join(df_sorted["Ticker"].head(15)))

# =============================
# STRATEGY GUIDE
# =============================
print("\n" + "="*60)
print("📘 OPTIONS STRATEGY EXECUTION GUIDE")
print("="*60)

print("""
ENTRY:
- Sell weekly put (~0.20 delta initially)
- Use defined risk: $2.5 or $5 wide spread (~8–10 weeks out)

SPREAD SELECTION:
- DEFAULT = $2.5 spread
- Upgrade to $5 ONLY if:
    ✔ Strong trend
    ✔ Clean earnings reaction
    ✔ Credit comfortably ≥ 10–15%

IMPORTANT:
- After Week 1:
  → DO NOT blindly follow 0.20 delta
  → Maintain risk structure
  → Only roll if credit justifies it

WEEKLY MANAGEMENT:

If price stays above strike:
→ Let expire
→ Sell next week

If price rises:
→ DO NOT aggressively move strike up
→ Only adjust if:
    ✔ Credit ≥ threshold
    ✔ Risk does NOT increase
    ✔ Strike remains ≤ 0.20 delta  ← (CRITICAL RULE)

If price drops near strike:
→ Roll ONLY if:
    ✔ Credit ≥ 10–15% of spread

CREDIT RULE (MANDATORY):
- Minimum credit:
    → ≥ 10% of spread width

- Preferred:
    → 12–16%

Examples:
- $2.5 spread → min $0.25
- $5 spread → min $0.50

❗ If below minimum:
→ SKIP trade

LONG PUT RULE:
- Do NOT auto-roll
- Only roll if:
    ✔ Total premium covers cost
    ✔ Position remains profitable

ASSIGNMENT:
- If assigned:
    → Sell long put (capture remaining value)
    → Close stock position

ONLY exercise if:
- No liquidity
- Near expiration (no time value)
- Emergency hedge needed

POSITION MANAGEMENT:
- Max 3 trades
- Add every ~3 weeks
- Avoid same sector overlap

CORE PRINCIPLE:
- Risk consistency > premium chasing
- Selectivity > frequency
- Skipping trades = edge
""")